# BananaClock ML Training Notebook (V11 - High Accuracy)

**Updates (V11): Tuned for higher accuracy.**
- **Unfrozen More Layers**: Top 50 layers (vs 30) for better feature learning.
- **Lower Learning Rate**: 3e-5 (vs 1e-4) for more stable fine-tuning.
- **Higher Patience**: 8 epochs (vs 5) to allow recovery from plateaus.
- **Stronger Regularization**: Increased Dropout to prevent overfitting.

## Prerequisites
1. **Datasets on Drive**: Ensure `My Drive/BananaClock/datasets/` contains these 3 files:
   - `Banana Ripeness Classification Dataset.zip`
   - `Banana Ripening Process.v2i.yolov8.zip`
   - `bdd69gyhv8-1.zip`
2. **Enable GPU**: Runtime → Change runtime type → T4 GPU

## Step 1: Check GPU and Install Dependencies

In [ ]:
# Check if GPU is available
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

!nvidia-smi

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python-headless pillow scikit-learn pandas matplotlib seaborn tqdm pyyaml

## Step 2: Mount Google Drive and Extract Datasets

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define Paths
DRIVE_BASE = "/content/drive/MyDrive/BananaClock/datasets"
LOCAL_BASE = "/content/datasets"

!mkdir -p {LOCAL_BASE}

In [ ]:
# Extract Zipped Datasets
import os
import glob

zip_files = [
    "Banana Ripeness Classification Dataset.zip",
    "Banana Ripening Process.v2i.yolov8.zip",
    "bdd69gyhv8-1.zip"
]

for zip_name in zip_files:
    path = f"{DRIVE_BASE}/{zip_name}"
    if os.path.exists(path):
        print(f"Extracting {zip_name}...")
        !unzip -qn "{path}" -d "{LOCAL_BASE}"
        print("Done.")
    else:
        print(f"ERROR: Could not find {path}")

print("\nExtracted contents:")
!ls -la {LOCAL_BASE}

## Step 3: Data Preparation (Optimized for 224x224)

Combine and preprocess all datasets. We now resize to **224x224** (Standard ResNet size) which is 4x faster than 416x416.

In [ ]:
%%writefile /content/prepare_datasets.py
"""
Dataset Preparation Script for BananaClock ML Training.
"""

import os
import argparse
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import logging
import yaml

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# OPTIMIZATION: 224x224 is standard for ResNet50
TARGET_SIZE = (224, 224)

CLASSES = ["fresh", "slightly_ripe", "ripe", "overripe", "spoiled"]

LABEL_MAPPING = {
    # Kaggle & General
    "unripe": "fresh", 
    "green": "fresh", 
    "freshripe": "ripe", # User confirmed yellow/ripe
    "ripe": "ripe", 
    "overripe": "overripe", 
    "rotten": "spoiled", 

    # Mendeley
    "freshbanana": "ripe",
    "rottenbanana": "spoiled",

    # YOLO
    "freshunripe": "fresh",
    "freshripe": "ripe",
}

def normalize_label(label):
    label_lower = label.lower().strip().replace(" ", "_").replace("-", "_")
    if label_lower in CLASSES:
        return label_lower
    if label_lower in LABEL_MAPPING:
        return LABEL_MAPPING[label_lower]
    return None

def process_image(image_path, output_path, crop_box=None, size=TARGET_SIZE):
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            if crop_box:
                w, h = img.size
                img = img.crop((
                    max(0, int(crop_box[0] * w)),
                    max(0, int(crop_box[1] * h)),
                    min(w, int(crop_box[2] * w)),
                    min(h, int(crop_box[3] * h))
                ))
            img = img.resize(size, Image.Resampling.LANCZOS)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            img.save(output_path, "JPEG", quality=95)
            return True
    except Exception:
        return False

def load_kaggle_dataset(dataset_dir):
    records = []
    dataset_dir = Path(dataset_dir)
    for split in ["train", "valid", "test"]:
        split_dir = dataset_dir / split
        if not split_dir.exists(): continue
        for class_dir in split_dir.iterdir():
            if class_dir.is_dir():
                label = normalize_label(class_dir.name)
                if label:
                    for img in class_dir.glob("*.*"):
                        records.append({"source": "kaggle", "original_path": str(img), "label": label, "split": split})
    logger.info(f"Kaggle: {len(records)} images")
    return pd.DataFrame(records)

def load_mendeley_dataset(dataset_dir):
    records = []
    dataset_dir = Path(dataset_dir)
    for sub in ["Original Image", "Augmented Image"]:
        sub_path = dataset_dir / sub
        if not sub_path.exists(): continue
        for fruit_dir in sub_path.iterdir():
            if fruit_dir.is_dir() and fruit_dir.name in ["FreshBanana", "RottenBanana"]:
                label = normalize_label(fruit_dir.name)
                if label:
                    for img in fruit_dir.glob("*.*"):
                        records.append({"source": "mendeley", "original_path": str(img), "label": label, "split": "train"})
    logger.info(f"Mendeley: {len(records)} images")
    return pd.DataFrame(records)

def load_yolo_dataset(dataset_dir):
    records = []
    dataset_dir = Path(dataset_dir)
    yaml_path = dataset_dir / "data.yaml"
    if not yaml_path.exists(): return pd.DataFrame(records)
    
    with open(yaml_path) as f: config = yaml.safe_load(f)
    names = config.get('names', {})
    if isinstance(names, list): id_to_name = {i: n for i, n in enumerate(names)}
    else: id_to_name = names

    for split in ['train', 'valid', 'test']:
        split_dir = dataset_dir / split if (dataset_dir/split).exists() else dataset_dir / ("val" if split=="valid" else split)
        if not split_dir.exists(): continue
        
        images_dir = split_dir / "images"
        labels_dir = split_dir / "labels"
        if not labels_dir.exists(): continue
        
        for label_file in labels_dir.glob("*.txt"):
            img_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                if (images_dir / (label_file.stem + ext)).exists():
                    img_path = images_dir / (label_file.stem + ext)
                    break
            if not img_path: continue

            with open(label_file) as lf:
                for line in lf:
                    parts = line.split()
                    cls_id = int(parts[0])
                    label_name = id_to_name.get(cls_id, str(cls_id))
                    label = normalize_label(label_name)
                    if not label: continue
                    xc, yc, w, h = map(float, parts[1:5])
                    records.append({
                        "source": "yolo", "original_path": str(img_path), "label": label, 
                        "split": split if split != "valid" else "val",
                        "crop": (xc-w/2, yc-h/2, xc+w/2, yc+h/2)
                    })
    logger.info(f"YOLO: {len(records)} crops")
    return pd.DataFrame(records)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--datasets-dir", type=str, default="/content/datasets")
    parser.add_argument("--output-dir", type=str, default="/content/processed_data")
    args = parser.parse_args()
    
    datasets_dir = Path(args.datasets_dir)
    output_dir = Path(args.output_dir)
    
    all_records = []
    all_records.append(load_kaggle_dataset(datasets_dir / "Banana Ripeness Classification Dataset"))
    all_records.append(load_mendeley_dataset(datasets_dir / "bdd69gyhv8-1"))
    all_records.append(load_yolo_dataset(datasets_dir / "Banana Ripening Process.v2i.yolov8"))
    
    df = pd.concat(all_records, ignore_index=True)
    if df.empty:
        print("ERROR: No images found!")
        return

    train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)
    
    for split, data in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"Processing {split} ({len(data)} images)...")
        # Using tqdm for progress checking
        for idx, row in tqdm(data.iterrows(), total=len(data)):
            dst = output_dir / split / row["label"] / f"{row['source']}_{idx}.jpg"
            crop = row.get("crop") if not pd.isna(row.get("crop")) else None
            process_image(row["original_path"], dst, crop_box=crop)

    print("\nDone! Final Statistics:")
    print(df["label"].value_counts())

if __name__ == "__main__":
    main()

In [ ]:
# Run Data Prep
!python /content/prepare_datasets.py

## Step 4: Train Classifier (ResNet50)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils import compute_class_weight

# Setup
DATA_DIR = "/content/processed_data"
OUTPUT_DIR = "/content/drive/MyDrive/BananaClock/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
# OPTIMIZATION: 224x224 matches native ResNet input, much faster
IMG_SIZE = (224, 224)
CLASSES = ["fresh", "slightly_ripe", "ripe", "overripe", "spoiled"]

# Generators with Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, brightness_range=[0.8, 1.2], fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=True
)
val_generator = val_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=False
)

# Class Weights
weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_generator.classes), 
    y=train_generator.classes
)
class_weights = dict(enumerate(weights))
print("Class Weights:", class_weights)

# Model Building
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# TUNING: Unfreeze more layers (50 instead of 30) for better accuracy
for layer in base_model.layers[:-50]: layer.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x) # Increased dropout
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x) # Increased dropout
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

model = keras.Model(inputs, outputs)

# TUNING: Lower learning rate for stable fine-tuning
optimizer = keras.optimizers.Adam(learning_rate=3e-5)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Training
model.fit(
    train_generator, validation_data=val_generator, epochs=50,
    callbacks=[
        # TUNING: Increased patience to 8
        EarlyStopping(patience=8, restore_best_weights=True),
        ModelCheckpoint(f"{OUTPUT_DIR}/banana_classifier_best.h5", save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ],
    class_weight=class_weights
)
model.save(f"{OUTPUT_DIR}/banana_classifier.h5")

## Step 5: Train YOLO Model

In [ ]:
import os
import shutil

OUTPUT_DIR = "/content/drive/MyDrive/BananaClock/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
from ultralytics import YOLO
import yaml

YOLO_DATA_PATH = "/content/datasets/Banana Ripening Process.v2i.yolov8/data.yaml"

# Fix paths in yaml
with open(YOLO_DATA_PATH) as f: config = yaml.safe_load(f)
base = os.path.dirname(YOLO_DATA_PATH)
config['train'] = f"{base}/train/images"
config['val'] = f"{base}/valid/images"
config['test'] = f"{base}/test/images"

with open("/content/yolo_config.yaml", "w") as f: yaml.dump(config, f)

# Train
model = YOLO('yolov8m.pt')
model.train(
    data="/content/yolo_config.yaml", epochs=100, imgsz=416, 
    project=OUTPUT_DIR, name='banana_yolo', patience=20
)

# Move best model
shutil.copy(f"{OUTPUT_DIR}/banana_yolo/weights/best.pt", f"{OUTPUT_DIR}/banana_yolo.pt")